In [1]:
import pandas as pd
import numpy as np
import sqlalchemy as sa

from sqlalchemy import create_engine
from pathlib import Path
import warnings
import time

warnings.filterwarnings("ignore")

# =========================================================
# DATABASE CONFIGURATION
# =========================================================

DB_USER = "postgres"

DB_PASSWORD = "1234"

DB_HOST = "localhost"

DB_PORT = "5433"

DB_NAME = "hr_analytics"

# =========================================================
# DATABASE ENGINE
# =========================================================

engine = create_engine(

    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}",

    pool_size=5,
    max_overflow=10,
    pool_timeout=30,
    pool_recycle=1800

)

print(" PostgreSQL connection established")

# =========================================================
# PROJECT PATHS
# =========================================================

BASE_PATH = Path.cwd().parent

RAW_DATA_PATH = BASE_PATH / "data" / "raw"

# =========================================================
# LOAD CSV FILES
# =========================================================

attrition_df = pd.read_csv(
    RAW_DATA_PATH / "attrition_raw.csv"
)

recruitment_df = pd.read_csv(
    RAW_DATA_PATH / "recruitment_raw.csv"
)

print("\n Raw datasets loaded")

# =========================================================
# ADVANCED COLUMN STANDARDIZATION
# =========================================================

def standardize_columns(df):

    df.columns = (

        df.columns
        .str.strip()
        .str.replace(" ", "_")
        .str.replace(r"([a-z])([A-Z])", r"\1_\2", regex=True)
        .str.lower()

    )

    return df

attrition_df = standardize_columns(attrition_df)

recruitment_df = standardize_columns(recruitment_df)

print("\n Standardized columns completed")

# =========================================================
# VALIDATION
# =========================================================

print("\nAttrition Shape:", attrition_df.shape)

print("Recruitment Shape:", recruitment_df.shape)

print("\nMissing Values")

print(attrition_df.isnull().sum().sum())

print(recruitment_df.isnull().sum().sum())

# =========================================================
# CREATE TABLES AUTOMATICALLY
# =========================================================

start_time = time.time()

attrition_df.to_sql(

    "employees",

    engine,

    if_exists="replace",

    index=False,

    method="multi",

    chunksize=1000

)

print(f"\n Employees table created")

recruitment_df.to_sql(

    "recruitment",

    engine,

    if_exists="replace",

    index=False,

    method="multi",

    chunksize=1000

)

print(f"\n Recruitment table created")

# =========================================================
# DATABASE VERIFICATION
# =========================================================

with engine.connect() as connection:

    employee_count = connection.execute(

        sa.text("SELECT COUNT(*) FROM employees")

    ).scalar()

    recruitment_count = connection.execute(

        sa.text("SELECT COUNT(*) FROM recruitment")

    ).scalar()

print("\n DATABASE VERIFICATION")

print(f"Employees Rows: {employee_count}")

print(f"Recruitment Rows: {recruitment_count}")

# =========================================================
# PREVIEW
# =========================================================

print("\n Employees Preview")

print(attrition_df.head(3))

print("\n Recruitment Preview")

print(recruitment_df.head(3))

print("\n Data pipeline completed successfully")

 PostgreSQL connection established

 Raw datasets loaded

 Standardized columns completed

Attrition Shape: (1470, 35)
Recruitment Shape: (1500, 25)

Missing Values
0
3402

 Employees table created

 Recruitment table created

 DATABASE VERIFICATION
Employees Rows: 1470
Recruitment Rows: 1500

 Employees Preview
   age attrition    business_travel  daily_rate              department  \
0   41       Yes      Travel_Rarely        1102                   Sales   
1   49        No  Travel_Frequently         279  Research & Development   
2   37       Yes      Travel_Rarely        1373  Research & Development   

   distance_from_home  education education_field  employee_count  \
0                   1          2   Life Sciences               1   
1                   8          1   Life Sciences               1   
2                   2          2           Other               1   

   employee_number  ...  relationship_satisfaction standard_hours  \
0                1  ...                    